# Milestone 4 - MCQ Task Formulation & LoRA Fine-Tuning Setup (WIP)

Setting up data formatting and sequence classification fine-tuning for HuggingFace transformer backbones.

In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset

data_path = '../guides/train.csv' if os.path.exists('../guides/train.csv') else 'train.csv'
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} samples for training setup.")
print("CUDA GPU available:", torch.cuda.is_available())

### MCQ Task Input Formatting & Label Mapping
Concatenating prompt text with all 5 option choices into a single input string and mapping letter answers A-E to integer IDs 0-4.

In [ ]:
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

def format_mcq_row(row):
    opts = f"(A) {row['A']} (B) {row['B']} (C) {row['C']} (D) {row['D']} (E) {row['E']}"
    return f"{row['prompt']} {opts}"

df['formatted_text'] = df.apply(format_mcq_row, axis=1)
df['label'] = df['answer'].map(LABEL_MAP)

print("Formatted Sample 0:", df['formatted_text'].iloc[0][:120], "...")
print("Sample 0 Target Label:", df['label'].iloc[0])

### LoRA Config & Fine-Tuning Setup (WIP)
Configuring `peft` LoRA adapters and `TrainingArguments` for `deberta-v3-small` sequence classification.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, TrainingArguments

# LoRA adapter configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["query_proj", "value_proj"]
)

model_name = "microsoft/deberta-v3-small"
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
lora_model = get_peft_model(base_model, peft_config)
lora_model.print_trainable_parameters()

# Training Arguments configuration
training_args = TrainingArguments(
    output_dir="./lora_deberta_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch"
)

# WARNING: batch_size=16 triggers CUDA OOM on local GPU setup during initial test; need to reduce batch size to 8.